# Day 3: Baseline ML — Vietnamese Price Prediction

**Dataset:** `SeanSunny/items_tv_v6` (110K train / 5K val / 5K test)

**Metrics:** RMSLE (primary), MAE (VND), MAPE (%), R2

| Step | Noi dung |
|------|----------|
| 3 | Baselines: Random, Mean, Median |
| 4 | LR + TF-IDF — Architecture A (n-gram) |
| 5 | LR + TF-IDF — Architecture B (underthesea) |
| 6 | Ensemble: RF, XGBoost, LightGBM, CatBoost |

In [ ]:
import random
import time
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor
from pricer_vi.items import Item
from pricer_vi.evaluator import evaluate

SEED = 42
DATASET = "SeanSunny/items_tv_v6"

## 1. Load Data

In [ ]:
train, val, test = Item.from_hub(DATASET)
print(f"Loaded {len(train):,} train, {len(val):,} val, {len(test):,} test items")
print(f"Sample: {test[0]}")
print(f"\nSummary: {test[0].summary[:300]}")

In [ ]:
train_prices = [item.price for item in train]
prices = np.array(train_prices, dtype=float)
documents = [item.summary for item in train]

print(f"Price range: {min(train_prices):,} - {max(train_prices):,} VND")
print(f"Mean price: {np.mean(train_prices):,.0f} VND")
print(f"Median price: {np.median(train_prices):,.0f} VND")
print(f"Std price: {np.std(train_prices):,.0f} VND")

---
## Step 3: Baselines (Random, Mean, Median)

In [ ]:
min_price = min(train_prices)
max_price = max(train_prices)

def random_pricer(item):
    return random.randint(min_price, max_price)

random.seed(SEED)
np.random.seed(SEED)
results_random = evaluate(random_pricer, test)

In [ ]:
training_average = sum(train_prices) / len(train_prices)
print(f"Training average price: {training_average:,.0f} VND")

def mean_pricer(item):
    return training_average

results_mean = evaluate(mean_pricer, test)

In [ ]:
training_median = float(np.median(train_prices))
print(f"Training median price: {training_median:,.0f} VND")

def median_pricer(item):
    return training_median

results_median = evaluate(median_pricer, test)

---
## Step 4: LR + TF-IDF — Architecture A (n-gram, khong tach tu)

- `TfidfVectorizer(max_features=5000, ngram_range=(1,2))`
- Split theo khoang trang mac dinh
- Bigram tu dong bat cum tu nhu "may tinh", "dien thoai"

In [ ]:
np.random.seed(SEED)
t0 = time.time()

vectorizer_a = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_a = vectorizer_a.fit_transform(documents)
print(f"TF-IDF A: {X_train_a.shape[0]:,} docs x {X_train_a.shape[1]:,} features")
print(f"Vectorization time: {time.time() - t0:.1f}s")

print(f"\nSample features: {list(vectorizer_a.get_feature_names_out()[2500:2520])}")

In [ ]:
t0 = time.time()
lr_model_a = LinearRegression()
lr_model_a.fit(X_train_a, prices)
print(f"Training time: {time.time() - t0:.1f}s")

def lr_tfidf_arch_a(item):
    x = vectorizer_a.transform([item.summary])
    return max(lr_model_a.predict(x)[0], 0)

results_lr_a = evaluate(lr_tfidf_arch_a, test)

---
## Step 5: LR + TF-IDF — Architecture B (underthesea pre-tokenize)

- `underthesea.word_tokenize()` ghep tu tieng Viet: "may tinh" -> "may_tinh"
- Multiprocessing 4 workers
- TF-IDF tren text da tokenize (unigram la du vi tu da ghep)

In [ ]:
from underthesea import word_tokenize
from multiprocessing import Pool

def tokenize_one(text):
    return word_tokenize(text, format="text")

np.random.seed(SEED)
t0 = time.time()
print(f"Pre-tokenizing {len(documents):,} train documents (4 workers)...")
with Pool(4) as p:
    tokenized_train = p.map(tokenize_one, documents)
print(f"Tokenization time: {time.time() - t0:.1f}s")

print(f"\nOriginal:  {documents[0][:150]}")
print(f"Tokenized: {tokenized_train[0][:150]}")

In [ ]:
t0 = time.time()
vectorizer_b = TfidfVectorizer(max_features=5000)
X_train_b = vectorizer_b.fit_transform(tokenized_train)
print(f"TF-IDF B: {X_train_b.shape[0]:,} docs x {X_train_b.shape[1]:,} features")
print(f"Vectorization time: {time.time() - t0:.1f}s")

print(f"\nSample features: {list(vectorizer_b.get_feature_names_out()[2500:2520])}")

In [ ]:
t0 = time.time()
lr_model_b = LinearRegression()
lr_model_b.fit(X_train_b, prices)
print(f"LR Training time: {time.time() - t0:.1f}s")

# Pre-tokenize test set (underthesea is NOT thread-safe)
t0 = time.time()
test_summaries = [item.summary for item in test]
print(f"Pre-tokenizing {len(test_summaries):,} test documents...")
with Pool(4) as p:
    tokenized_test = p.map(tokenize_one, test_summaries)
tokenized_test_map = {item.summary: tok for item, tok in zip(test, tokenized_test)}
print(f"Test tokenization time: {time.time() - t0:.1f}s")

In [ ]:
def lr_tfidf_arch_b(item):
    if item.summary in tokenized_test_map:
        tokenized = tokenized_test_map[item.summary]
    else:
        tokenized = tokenize_one(item.summary)
    x = vectorizer_b.transform([tokenized])
    return max(lr_model_b.predict(x)[0], 0)

results_lr_b = evaluate(lr_tfidf_arch_b, test)

---
## Step 6: Ensemble Models (dung Architecture B)

Tree-based models khong predict gia am -> RMSLE se tot hon LR.

### 6a. Random Forest (subset 20K)

In [ ]:
np.random.seed(SEED)
SUBSET_RF = 20_000
t0 = time.time()
rf_model = RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=4)
rf_model.fit(X_train_b[:SUBSET_RF], prices[:SUBSET_RF])
print(f"Training time: {time.time() - t0:.1f}s (on {SUBSET_RF:,} samples)")

def random_forest_pricer(item):
    if item.summary in tokenized_test_map:
        tokenized = tokenized_test_map[item.summary]
    else:
        tokenized = tokenize_one(item.summary)
    x = vectorizer_b.transform([tokenized])
    return max(0, rf_model.predict(x)[0])

results_rf = evaluate(random_forest_pricer, test)

### 6b. XGBoost (full data)

In [ ]:
np.random.seed(SEED)
t0 = time.time()
xgb_model = xgb.XGBRegressor(
    n_estimators=1000, learning_rate=0.1, random_state=SEED, n_jobs=4
)
xgb_model.fit(X_train_b, prices)
print(f"Training time: {time.time() - t0:.1f}s")

def xgboost_pricer(item):
    if item.summary in tokenized_test_map:
        tokenized = tokenized_test_map[item.summary]
    else:
        tokenized = tokenize_one(item.summary)
    x = vectorizer_b.transform([tokenized])
    return max(0, xgb_model.predict(x)[0])

results_xgb = evaluate(xgboost_pricer, test)

### 6c. LightGBM (full data)

In [ ]:
np.random.seed(SEED)
t0 = time.time()
lgb_model = lgb.LGBMRegressor(
    n_estimators=1000, learning_rate=0.1, random_state=SEED, n_jobs=4, verbose=-1
)
lgb_model.fit(X_train_b, prices)
print(f"Training time: {time.time() - t0:.1f}s")

def lightgbm_pricer(item):
    if item.summary in tokenized_test_map:
        tokenized = tokenized_test_map[item.summary]
    else:
        tokenized = tokenize_one(item.summary)
    x = vectorizer_b.transform([tokenized])
    return max(0, lgb_model.predict(x)[0])

results_lgb = evaluate(lightgbm_pricer, test)

### 6d. CatBoost (full data)

In [ ]:
np.random.seed(SEED)
t0 = time.time()
cb_model = CatBoostRegressor(
    iterations=1000, learning_rate=0.1, random_seed=SEED, verbose=0
)
cb_model.fit(X_train_b, prices)
print(f"Training time: {time.time() - t0:.1f}s")

def catboost_pricer(item):
    if item.summary in tokenized_test_map:
        tokenized = tokenized_test_map[item.summary]
    else:
        tokenized = tokenize_one(item.summary)
    x = vectorizer_b.transform([tokenized])
    return max(0, cb_model.predict(x)[0])

results_cb = evaluate(catboost_pricer, test)

---
## Final Summary

In [ ]:
summary = pd.DataFrame([
    {"Model": "Random", **results_random},
    {"Model": "Mean", **results_mean},
    {"Model": "Median", **results_median},
    {"Model": "LR + TF-IDF (Arch A)", **results_lr_a},
    {"Model": "LR + TF-IDF (Arch B)", **results_lr_b},
    {"Model": "Random Forest (Arch B)", **results_rf},
    {"Model": "XGBoost (Arch B)", **results_xgb},
    {"Model": "LightGBM (Arch B)", **results_lgb},
    {"Model": "CatBoost (Arch B)", **results_cb},
])
summary = summary.set_index("Model")
summary.style.format({
    "rmsle": "{:.4f}",
    "mae": "{:,.0f}",
    "mape": "{:.1f}",
    "r2": "{:.1f}",
}).highlight_min(subset=["rmsle", "mae", "mape"], color="lightgreen").highlight_max(subset=["r2"], color="lightgreen")